In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms 

**MLP / `nn.Module` essentials — the must-not-forget list**

**1. Without nonlinearity, depth is fake**
Stacked linear layers collapse to one linear layer. ReLU (or any nonlinearity) between them is what makes depth meaningful. If you ever write a model with no activations between Linear layers, you've built a more expensive single Linear layer.

**2. Every PyTorch model follows the same `nn.Module` pattern**
- Subclass `nn.Module`
- `super().__init__()` first line of `__init__` (always)
- Declare sub-modules as attributes in `__init__`
- Describe data flow in `forward`

CNN, RNN, Transformer, TCN — same scaffold, different sub-modules. Learn the pattern once, reuse forever.

**3. `super().__init__()` is mandatory**
If you skip it, parameter registration breaks silently. `model.parameters()` returns empty, optimizer has nothing to update, training appears to run but learns nothing. Easy to forget, painful to debug.

**4. Sub-modules assigned as attributes auto-register**
`self.fc1 = nn.Linear(...)` automatically adds `fc1.weight` and `fc1.bias` to `model.parameters()`. This is why the optimizer just works. If you put modules in a plain Python list, they don't register — use `nn.ModuleList` or `nn.Sequential` instead.

**5. Always call modules, never `.forward()` directly**
`self.fc1(x)` not `self.fc1.forward(x)`. Recursively, all the way down. The `__call__` wrapper handles hooks and train/eval state.

**6. Logits ≠ probabilities ≠ class labels**
- Logits: raw model output, unbounded real numbers
- Probabilities: `softmax(logits)`, in [0, 1], sum to 1
- Class label: `argmax(logits)` (= argmax of probabilities)

For training classification, output logits and let `CrossEntropyLoss` handle softmax internally. Applying softmax yourself before the loss is a bug.

**7. Loss function dictates output format**
- Regression → MSE, output raw values
- Binary classification → BCEWithLogitsLoss, output 1 logit per sample
- Multi-class classification → CrossEntropyLoss, output K logits per sample (K = num classes)

Choose the loss first, then design the output layer to match.

**8. The funnel principle**
Hidden layers progressively compress toward the output. Each layer forces the network to discard pixel-level noise and keep task-relevant features. This is the **information bottleneck** view of representation learning — applies to every architecture, not just MLPs.

**9. Architecture choices are mostly judgment + empirics**
- Input size: dictated by data
- Output size: dictated by task
- Depth: 2-3 hidden layers for MLPs; deeper needs residuals/normalization to train
- Width: scales with problem complexity, tune empirically
- Powers of 2 are convention, not magic

If train accuracy is low → model too small. If train >> val → too big or needs regularization.

**10. ReLU is the default activation between hidden layers**
No activation on the output layer for classification (logits), no activation on the output layer for regression (raw values). Activation only between hidden layers.

**11. The training loop is identical to Step 1, just nested**
for epoch:
for batch:
zero_grad → forward → loss → backward → step

The five-beat rhythm doesn't change. You just iterate over batches now instead of the whole dataset at once.

**12. Inference pattern: `model.eval()` + `torch.no_grad()`**
Both, together, every time you're not training:
- `model.eval()` switches dropout/batchnorm to inference mode
- `torch.no_grad()` disables gradient tracking to save memory

Forget either and you get subtle bugs (eval mode wrong) or wasted memory (no_grad missing).

**13. `DataLoader` handles batching, shuffling, and iteration**
You give it a `Dataset`, it gives you batches. `shuffle=True` for train, `shuffle=False` for test/val. Batch size 32-256 is typical; bigger = faster per epoch but more memory and sometimes worse generalization.

**14. Shape discipline**
Print `.shape` whenever something feels off. For MLPs the input shape after flatten is `(batch, features)`. Mismatches between expected and actual shapes is the #1 cause of cryptic errors.

---

**The one-liner upgrade from Step 1:**

> Step 1: PyTorch tracks gradients by default; the training loop is zero-forward-loss-backward-step.
>
> Step 2: Every model is an `nn.Module` subclass with sub-modules in `__init__` and data flow in `forward`. Loss function dictates output format. Depth without nonlinearity is fake depth.

In [2]:
# transform: convert pil image to tensor and normalize to [0, 1] 
transform = transforms.Compose([
    transforms.ToTensor(), # converts (H, W) PIL -> (1, 28, 28) tensor in [0, 1]
]) 
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)


100%|██████████| 26.4M/26.4M [00:20<00:00, 1.26MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 126kB/s]
100%|██████████| 4.42M/4.42M [00:07<00:00, 582kB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


In [3]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader)) 
print(f"Batch shape: {images.shape}")  # (64, 1, 28, 28)
print(f"Labels shape: {labels.shape}")  # (64,)
print(f"Label range: {labels.min()} to {labels.max()}")  # 0 to 9 

Batch shape: torch.Size([64, 1, 28, 28])
Labels shape: torch.Size([64])
Label range: 0 to 9


In [16]:
# create the model 
# Input → Linear → Activation → Linear → Activation → ... → Linear → Output 

class MLP(nn.Module): 
    def __init__(self): 
        super().__init__() 
        self.flatten = nn.Flatten() # (B, 1, 28, 28) -> (B, 784) 
        self.fc1 = nn.Linear(28 * 28, 128) # (input_shape, output_shape) -> (784, 128) 
        self.fc2 = nn.Linear(128, 64) 
        self.fc3 = nn.Linear(64, 10) # output logits 

    def forward(self, x):
        x = self.flatten(x) # flatten the image first 
        x = torch.relu(self.fc1(x)) # activation(linear) 
        x = torch.relu(self.fc2(x)) 
        x = self.fc3(x) # raw logits - no softmax  
        # nn.CrossEntropyLoss applies softmax internally for numerical stability
        return x 

model = MLP() 
print(model)

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


In [18]:
out = model(images)
print(f"Output shape: {out.shape}")  # (64, 10)
print(out)

Output shape: torch.Size([16, 10])
tensor([[ 0.0460, -0.0784, -0.0594, -0.1296, -0.0433,  0.0011,  0.0575, -0.0219,
         -0.0155, -0.0241],
        [ 0.0766, -0.0777, -0.0364, -0.1186, -0.0347, -0.0310,  0.0296, -0.0193,
          0.0252, -0.0290],
        [ 0.0298, -0.1051, -0.0374, -0.1152, -0.0589, -0.0260,  0.0432, -0.0519,
         -0.0048, -0.0432],
        [ 0.0239, -0.0641, -0.0627, -0.1468, -0.0382, -0.0377,  0.0656,  0.0127,
         -0.0127, -0.0427],
        [ 0.0532, -0.0664, -0.0709, -0.1181, -0.0394, -0.0301,  0.0511, -0.0184,
          0.0122, -0.0982],
        [ 0.0365, -0.0664, -0.0931, -0.1339, -0.0127, -0.0361,  0.0243, -0.0369,
          0.0297, -0.0524],
        [ 0.0644, -0.0571, -0.0597, -0.1752,  0.0037, -0.0522,  0.0864,  0.0616,
         -0.0059, -0.0772],
        [ 0.0739, -0.0794, -0.0815, -0.1078, -0.0087, -0.0346,  0.0846, -0.0489,
          0.0063, -0.0311],
        [ 0.0267, -0.0613, -0.0450, -0.1427, -0.0849, -0.0017,  0.0655, -0.0336,
         -0.

In [ ]:
loss_fn = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=1e-3)  
# adapotive moment estimation. 
n_epochs = 5 
for epoch in range(n_epochs): 
    model.train() 
    running_loss = 0.0 
    correct = 0 
    total = 0 
    for images, labels in train_loader: 
        optimizer.zero_grad() 
        logits = model(images) 
        loss = loss_fn(logits, labels) 
        loss.backward() 
        optimizer.step() 

        # track the loss 
        running_loss += loss.item() 
        preds = logits.argmax(dim=1) 
        correct += (preds == labels).sum().item() 
        total += labels.size(0) 
    train_acc = correct / total 
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}: loss={avg_loss:.4f}, train_acc={train_acc:.4f}") 

Epoch 1: loss=0.5625, train_acc=0.8025
Epoch 2: loss=0.3950, train_acc=0.8583
Epoch 3: loss=0.3518, train_acc=0.8719
Epoch 4: loss=0.3235, train_acc=0.8815
Epoch 5: loss=0.3050, train_acc=0.8879


In [20]:
# eval 
model.eval() 
correct = 0 
total = 0 
with torch.no_grad(): 
    for images, labels in test_loader: 
        logits = model(images)
        preds = logits.argmax(dim=1) 
        correct += (preds == labels).sum().item() 
        total += labels.size(0) 

print(f"Test accuracy: {correct / total}:.4f")

Test accuracy: 0.8689:.4f
